# Airbnb Paris – Exp 2: Enhanced Baseline-Modelle
- Gleiche Baselines auf verschiedenen Repräsentationen: cleaned vs. semantisch (PCA/no PCA) vs. fastText (PCA30/100) vs. enhanced (PCA/no PCA) vs. enhanced+semantisch
- Alignment über `row_id`, gemeinsamer 70/30-Split; feste Detektor-Params (fairer Vergleich)

In [13]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Repräsentationen laden (indexiert über row_id)
- Outlier = `is_top_rating == 0`; enhanced+semantisch = PCA30-Konkatenation

In [14]:
LABEL = "is_top_rating"
ds = "airbnb_paris"

def_load = lambda name: pd.read_csv(f"../../data/preprocessed/{name}_{ds}.csv").set_index("row_id")
cleaned = def_load("cleaned")
semantic_pca100 = def_load("semantic_pca100")
semantic_pca = def_load("semantic_pca30")
fast_text_pca100 = def_load("fast_text_pca100")
fast_text_pca30 = def_load("fast_text_pca30")
enhanced = def_load("enhanced")
enhanced_pca = def_load("enhanced_pca30")

reps = {
    "cleaned": cleaned.drop(columns=[LABEL]),
    "semantic_pca100": semantic_pca100.drop(columns=[LABEL]),
    "semantic_pca30": semantic_pca.drop(columns=[LABEL]),
    "fast_text_pca100": fast_text_pca100.drop(columns=[LABEL]),
    "fast_text_pca30": fast_text_pca30.drop(columns=[LABEL]),
    "enhanced": enhanced.drop(columns=[LABEL]),
    "enhanced_pca30": enhanced_pca.drop(columns=[LABEL]),
    "enhanced_semantic_pca30": enhanced_pca.drop(columns=[LABEL]).join(
        semantic_pca.drop(columns=[LABEL]), how="inner", lsuffix="_enh", rsuffix="_sem"),
}

## Gemeinsamer Index & Split
- Schnittmenge aller Repräsentationen (robust gegen unvollständige semantic-CSVs)

In [15]:
common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()
y = (1 - cleaned.loc[common, LABEL]).values
print("common rows:", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = (1 - cleaned.loc[tr_id, LABEL]).values
y_test = (1 - cleaned.loc[te_id, LABEL]).values

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_2")

common rows: 18350 outlier rate 0.0393


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/airbnb_notebooks/exp2/../../mlruns/230520600845878770', creation_time=1782742727671, experiment_id='230520600845878770', last_update_time=1782742727671, lifecycle_stage='active', name='airbnb_paris_experiment_2', tags={}, trace_location=None, workspace='default'>

## Detektoren x Repräsentationen
- Beste Hyperparameter aus Exp 1 (README), **kein GridSearch**; AutoEncoder unsupervised (Originalverteilung, GPU)

In [16]:
# beste Hyperparameter aus Experiment 1 (README) — kein GridSearch in Exp 2
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "random_state": 42}, False),
    "loda": (LODA, {"n_bins": 10, "n_random_cuts": 100}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 50, "random_state": 42, "device": "cuda"}, False),
}

for rep_name, rep in reps.items():
    Xtr = rep.loc[tr_id].values
    Xte = rep.loc[te_id].values
    for det_name, (Model, params, inlier_only) in detectors.items():
        t0 = time.perf_counter()
        Xfit = Xtr[y_train == 0] if inlier_only else Xtr
        model = Model(**params)
        model.fit(Xfit)
        scores = model.decision_function(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_test, scores)
        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_param("representation", rep_name)
            mlflow.log_param("detector", det_name)
            mlflow.log_param("n_features", rep.shape[1])
            mlflow.log_metric("average_precision", ap)
            mlflow.log_metric("auprc", auprc)
            mlflow.log_metric("auc_roc", auroc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")

cleaned                  iforest      AP=0.0886 AUPRC=0.0867 AUC=0.6962 feat=40 t=0.4s
cleaned                  loda         AP=0.0951 AUPRC=0.0933 AUC=0.6825 feat=40 t=0.1s
cleaned                  ecod         AP=0.1085 AUPRC=0.1058 AUC=0.7240 feat=40 t=0.2s


Training: 100%|██████████| 50/50 [00:56<00:00,  1.14s/it]


cleaned                  autoencoder  AP=0.0731 AUPRC=0.0707 AUC=0.6108 feat=40 t=57.5s
semantic_pca100          iforest      AP=0.0698 AUPRC=0.0675 AUC=0.6116 feat=154 t=0.3s
semantic_pca100          loda         AP=0.0514 AUPRC=0.0504 AUC=0.5150 feat=154 t=0.1s
semantic_pca100          ecod         AP=0.0677 AUPRC=0.0651 AUC=0.6141 feat=154 t=1.2s


Training: 100%|██████████| 50/50 [00:58<00:00,  1.17s/it]


semantic_pca100          autoencoder  AP=0.0455 AUPRC=0.0434 AUC=0.5009 feat=154 t=59.2s
semantic_pca30           iforest      AP=0.0806 AUPRC=0.0784 AUC=0.6629 feat=84 t=0.3s
semantic_pca30           loda         AP=0.0684 AUPRC=0.0657 AUC=0.5937 feat=84 t=0.1s
semantic_pca30           ecod         AP=0.0846 AUPRC=0.0821 AUC=0.6733 feat=84 t=0.5s


...

enhanced                 autoencoder  AP=0.2319 AUPRC=0.2296 AUC=0.7742 feat=512 t=62.4s
enhanced_pca30           iforest      AP=0.2305 AUPRC=0.2291 AUC=0.7569 feat=30 t=0.3s
enhanced_pca30           loda         AP=0.1937 AUPRC=0.1919 AUC=0.7299 feat=30 t=0.1s
enhanced_pca30           ecod         AP=0.2705 AUPRC=0.2697 AUC=0.7455 feat=30 t=0.2s


Training: 100%|██████████| 50/50 [00:56<00:00,  1.13s/it]


enhanced_pca30           autoencoder  AP=0.1315 AUPRC=0.1298 AUC=0.7148 feat=30 t=57.1s
enhanced_semantic_pca30  iforest      AP=0.1029 AUPRC=0.1012 AUC=0.7306 feat=114 t=0.3s
enhanced_semantic_pca30  loda         AP=0.0653 AUPRC=0.0627 AUC=0.5384 feat=114 t=0.1s
enhanced_semantic_pca30  ecod         AP=0.1462 AUPRC=0.1434 AUC=0.7620 feat=114 t=0.7s


Training: 100%|██████████| 50/50 [00:53<00:00,  1.08s/it]


enhanced_semantic_pca30  autoencoder  AP=0.1459 AUPRC=0.1424 AUC=0.6637 feat=114 t=54.5s
